In [10]:
import pandas as pd
import numpy as np

In [11]:
df = pd.read_csv("data.csv")
df.head()

,Unnamed: 0.1,Unnamed: 0,brand,name,price,spec_rating,processor,CPU,Ram,Ram_type,ROM,ROM_type,GPU,display_size,resolution_width,resolution_height,OS,warranty
0,0,0,HP,Victus 15-fb0157AX Gaming Laptop,49900,73.000000,5th Gen AMD Ryzen 5 5600H,"Hexa Core, 12 Threads",8GB,DDR4,512GB,SSD,4GB AMD Radeon RX 6500M,15.6,1920.0,1080.0,Windows 11 OS,1
1,1,1,HP,15s-fq5007TU Laptop,39900,60.000000,12th Gen Intel Core i3 1215U,"Hexa Core (2P + 4E), 8 Threads",8GB,DDR4,512GB,SSD,Intel UHD Graphics,15.6,1920.0,1080.0,Windows 11 OS,1
2,2,2,Acer,One 14 Z8-415 Laptop,26990,69.323529,11th Gen Intel Core i3 1115G4,"Dual Core, 4 Threads",8GB,DDR4,512GB,SSD,Intel Iris Xe Graphics,14.0,1920.0,1080.0,Windows 11 OS,1
3,3,3,Lenovo,Yoga Slim 6 14IAP8 82WU0095IN Laptop,59729,66.000000,12th Gen Intel Core i5 1240P,"12 Cores (4P + 8E), 16 Threads",16GB,LPDDR5,512GB,SSD,Intel Integrated Iris Xe,14.0,2240.0,1400.0,Windows 11 OS,1
4,4,4,Apple,MacBook Air 2020 MGND3HN Laptop,69990,69.323529,Apple M1,Octa Core (4P + 4E),8GB,DDR4,256GB,SSD,Apple M1 Integrated Graphics,13.3,2560.0,1600.0,Mac OS,1


In [12]:
# Keep only needed columns
df = df[["brand", "CPU", "Ram", "ROM", "price"]].copy()
df.isnull().sum()

brand    0
CPU      0
Ram      0
ROM      0
price    0
dtype: int64

In [13]:
# Convert RAM/ROM to numeric GB
df["Ram_GB"] = df["Ram"].astype(str).str.extract(r"(\d+)").astype(float)
df["ROM_GB"] = df["ROM"].astype(str).str.extract(r"(\d+)").astype(float)
df = df.drop(columns=["Ram", "ROM"])
df.head()

,brand,CPU,price,Ram_GB,ROM_GB
0,HP,"Hexa Core, 12 Threads",49900,8.0,512.0
1,HP,"Hexa Core (2P + 4E), 8 Threads",39900,8.0,512.0
2,Acer,"Dual Core, 4 Threads",26990,8.0,512.0
3,Lenovo,"12 Cores (4P + 8E), 16 Threads",59729,16.0,512.0
4,Apple,Octa Core (4P + 4E),69990,8.0,256.0


In [14]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

In [15]:
features = ["brand", "CPU", "Ram_GB", "ROM_GB"]
X = df[features]
y = df["price"]

numeric_features = ["Ram_GB", "ROM_GB"]
categorical_features = ["brand", "CPU"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [16]:
numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features),
    ]
)

model = RandomForestRegressor(
    n_estimators=300,
    random_state=42
)

regressor = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        ("model", model),
    ]
)

In [17]:
# Train
regressor.fit(X_train, y_train)

# Evaluate
y_pred = regressor.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("MAE:", mae)
print("R2:", r2)

MAE: 16576.797055387466
R2: 0.7707507421087448


In [18]:
# Save full pipeline
joblib.dump(regressor, "rf_laptop_price_model.pkl")

['rf_laptop_price_model.pkl']